In [12]:
# %% [markdown]
# # Legacy CSV Import - Step 1: Read / Parse / Print / Export payload
# 
# 输入目录：
#   ./Old_data_csv/*.csv
# 输出：
#   ./import_payload.pkl   (给 Notebook 2 使用)
#
# 注意：
# - 每处理 1 条 entry 都会 print，方便你定位哪条出问题
# - 所有条目处理完后一次性保存 payload（不写 DB）

# %%
import os
import re
import csv
import pickle
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime, timezone
from typing import List, Dict, Any, Optional, Tuple

DATA_DIR = Path("./Old_data_csv")
OUT_PAYLOAD = Path("./import_payload.pkl")

# 允许的品牌（从文件名自动识别；这里只是用于规范化显示）
KNOWN_BRANDS = {"Chanel", "Dior", "Hermes", "LV", "LouisVuitton", "Louis_Vuitton", "Louis Vuitton"}

# 文件名模式：BrandYYYYSold.csv / BrandYYYYUnsold.csv
# 1) 带年份：BrandYYYY(Sold|Unsold).csv
FILENAME_RE_YEAR = re.compile(
    r"^(?P<brand>.+?)(?P<year>\d{4})(?P<kind>Sold|Unsold)\.csv$",
    re.I
)

# 2) 不带年份：BrandUnsold.csv  （只针对 Unsold）
FILENAME_RE_NOYEAR_UNSOLD = re.compile(
    r"^(?P<brand>.+?)Unsold\.csv$",
    re.I
)

# sold 行里可能出现的日期 token：
# - "25 20.05" / "20.05" / "20.05." / "25 20.05."
# 我们只取最后一个 dd.mm
SOLD_DATE_RE = re.compile(r"(\d{1,2})\.(\d{1,2})")

# 收货日期 token 容错：
# - "16.01" / "16.01."
# - "19,01"  -> 视为 19.01
# - "12.07/19.07" -> 取 / 前面的 12.07
# - "24 18.04" -> 取最后一个 dd.mm
RECV_DATE_RE = re.compile(r"(\d{1,2})[.,](\d{1,2})")

def parse_recv_day_month(token: str) -> Optional[Tuple[int, int]]:
    if token is None:
        return None

    s = str(token).strip()
    if not s:
        return None

    # 1) "12.07/19.07" -> 取前半
    if "/" in s:
        s = s.split("/", 1)[0].strip()

    # 2) 逗号当点
    s = s.replace(",", ".")

    # ⭐ 3) 移除点号前后的空格（04 .12 / 04. 12 / 04.12）
    s = re.sub(r"\s*\.\s*", ".", s)

    # 4) 在整个字符串中找 dd.mm
    hits = RECV_DATE_RE.findall(s)
    if not hits:
        return None

    day, month = hits[-1]
    day = int(day)
    month = int(month)

    if not (1 <= month <= 12 and 1 <= day <= 31):
        return None

    return day, month



def parse_sold_day_month(token: str) -> Optional[Tuple[int, int]]:
    if token is None:
        return None
    s = str(token).strip()
    # 找最后一个 dd.mm
    hits = SOLD_DATE_RE.findall(s)
    if not hits:
        return None
    day, month = hits[-1]
    day = int(day); month = int(month)
    if not (1 <= month <= 12 and 1 <= day <= 31):
        return None
    return day, month

def safe_dt_utc(year: int, month: int, day: int) -> Optional[datetime]:
    """
    构造 UTC datetime。
    特殊容错规则：
    - 如果是 2 月 29 日且非法（如非闰年），直接降级为 2 月 28 日
    """
    year = int(year)
    month = int(month)
    day = int(day)

    # ⭐ 简单特判：2 月 29 日 → 2 月 28 日
    if month == 2 and day == 29:
        day = 28

    try:
        return datetime(year, month, day, 0, 0, tzinfo=timezone.utc)
    except ValueError:
        return None



def to_int_maybe(x) -> Optional[int]:
    if x is None:
        return None
    s = str(x).strip()
    if s == "":
        return None
    # 有时可能是 "15,627" 这种
    s = s.replace(",", "")
    try:
        return int(float(s))
    except:
        return None

def normalize_brand(raw: str) -> str:
    b = str(raw).strip()
    # 你文件名里可能是 LV 或 LouisVuitton
    if b.lower() in ["louisvuitton", "louis_vuitton", "louis vuitton"]:
        return "Louis Vuitton"
    if b.lower() == "lv":
        return "Louis Vuitton"
    # Chanel/Dior/Hermes
    return b[:1].upper() + b[1:]

def sniff_rows(filepath: Path) -> List[List[str]]:
    """
    你的 CSV 看起来更像是 'tab 分隔' 或者 'Excel 导出后分隔不标准'。
    这里做一个稳健读取：
    - 先用 csv.Sniffer 判断 delimiter（常见为 \t 或 ,）
    - 如果 sniff 失败：优先按 \t 分割；否则退化为按多个空格分割
    """
    text = filepath.read_text(encoding="utf-8", errors="ignore")
    lines = [ln for ln in text.splitlines() if ln.strip() != ""]
    if not lines:
        return []

    sample = "\n".join(lines[:20])
    delimiter = None
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=[",", "\t", ";"])
        delimiter = dialect.delimiter
    except:
        delimiter = "\t"

    rows: List[List[str]] = []
    if delimiter in [",", "\t", ";"]:
        reader = csv.reader(lines, delimiter=delimiter)
        for r in reader:
            rows.append([c.strip() for c in r])
    else:
        # fallback
        for ln in lines:
            rows.append([c.strip() for c in re.split(r"\s{2,}", ln.strip())])

    # 有些“复制出来没有逗号”的情况，会导致整行变 1 列，我们再做一次二次分割：
    fixed = []
    for r in rows:
        if len(r) == 1:
            # 试试按 tab / 多空格
            parts = [c.strip() for c in r[0].split("\t") if c.strip() != ""]
            if len(parts) <= 1:
                parts = [c.strip() for c in re.split(r"\s{2,}", r[0].strip()) if c.strip() != ""]
            fixed.append(parts)
        else:
            fixed.append(r)
    return fixed

def concat_notes(a: Optional[str], b: Optional[str]) -> str:
    a = (a or "").strip()
    b = (b or "").strip()
    if a and b:
        return f"{a} | {b}"
    return a or b

def parse_sold_row(cols: List[str], brand: str, recv_year: int, file: str, row_idx: int) -> Optional[Dict[str, Any]]:
    """
    Sold 文件示例列（按你的描述）：
    0 recv_dd.mm
    1 name
    2 cost_rmb
    3 sell_sgd
    4 profit_rmb
    5 sold_date_token (可能 "25 20.05")
    6 sold_year (如 2025)
    7 serial_code (可能空)
    8 note1 (可能空)
    9 note2 (可能空)
    """
    # 容错：长度不够就补空
    while len(cols) < 10:
        cols.append("")

    recv_token = cols[0]
    name = cols[1]
    cost_rmb = to_int_maybe(cols[2])
    sell_sgd = to_int_maybe(cols[3])
    profit_rmb = to_int_maybe(cols[4])
    sold_token = cols[5]
    sold_year = to_int_maybe(cols[6])
    serial = (cols[7] or "").strip() or None
    note = concat_notes(cols[8], cols[9])

    dm = parse_recv_day_month(recv_token)
    if not dm:
        print(f"[WARN] {file} row#{row_idx}: invalid recv date token: {recv_token!r} -> skip")
        return None
    r_day, r_month = dm
    received_at = safe_dt_utc(recv_year, r_month, r_day)
    if not received_at:
        print(f"[WARN] {file} row#{row_idx}: invalid received date -> year={recv_year} token={recv_token!r} parsed={r_day}.{r_month} -> skip")
        return None


    sold_at = None
    if sold_year:
        sdm = parse_sold_day_month(sold_token)
        if sdm:
            s_day, s_month = sdm
            sold_at = safe_dt_utc(int(sold_year), s_month, s_day)
            if sold_at is None:
                print(f"[WARN] {file} row#{row_idx}: invalid sold date -> year={sold_year} token={sold_token!r} parsed={s_day}.{s_month} -> keep sold_at=None")

        else:
            # sold_year 有但 sold_day_month 解析不到
            print(f"[WARN] {file} row#{row_idx}: sold_year={sold_year} but sold_token={sold_token!r} not parseable")
    else:
        # 有些行可能 sold_year 为空；按你的说明以列为准，所以不强推
        if str(sold_token).strip():
            print(f"[WARN] {file} row#{row_idx}: sold_token present but sold_year empty -> keep sold_at=None")

    # 基础字段：你 app 的 items schema 可识别 + 我们额外加 legacy 字段
    item_doc = {
        # sku 在 Notebook2 生成，避免 DB 冲突
        "name": name.strip(),
        "brand": brand,
        "currency": "RMB",                  # 以成本币种为主
        "cost": int(cost_rmb or 0),         # 你系统里 cost 是 int
        "sell_price": int(sell_sgd or 0),   # 注意：这里 sell_price 数值是 SGD，但字段仍叫 sell_price（历史导入兼容）
        "sell_currency": "SGD",             # 新增字段标注
        "profit": int(profit_rmb or 0),     # 直接写利润（你要求）
        "profit_currency": "RMB",           # 新增字段标注
        "fee": 0,                           # 历史数据没拆费，设0
        "status": "SOLD",
        "note": note,
        "serial_code": serial,
        "code": "",                         # 旧表未提供，先空
        "accessories": "",                  # 旧表未提供，先空

        "seller_name": "",                  # 旧表没有明确字段
        "seller_contact": "",

        "source_type": "BUY_IN",            # 默认收货（你可后续手动修正）
        "is_buy_in": True,
        "is_consignment": False,

        "created_at": received_at,          # 让它看起来像当时建档
        "updated_at": received_at,
        "received_at": received_at,
        "sold_at": sold_at,

        "audit": [{
            "at": datetime.now(timezone.utc),
            "action": "IMPORT_LEGACY",
            "detail": {"file": file, "row": row_idx, "kind": "Sold"}
        }]
    }

    sale_doc = {
        # item_id / sku 在 Notebook2 填
        "buyer": "",
        "channel": "legacy",
        "sell_price": int(sell_sgd or 0),
        "fee": 0,
        "cost": int(cost_rmb or 0),         # 成本 RMB
        "profit": int(profit_rmb or 0),     # 利润 RMB
        "currency": "MIXED",                # 标注历史混币
        "note": note,
        "created_at": sold_at or received_at
    }

    # 打印给你检查
    print(f"[OK] {file} row#{row_idx} | {brand} | recv={received_at.date()} | sold={sold_at.date() if sold_at else None} | "
          f"costRMB={cost_rmb} sellSGD={sell_sgd} profitRMB={profit_rmb} | serial={serial or '-'} | name={name[:30]}...")

    return {"item": item_doc, "sale": sale_doc, "meta": {"file": file, "row": row_idx, "kind": "Sold"}}

def parse_unsold_row(cols: List[str], brand: str, recv_year_from_filename: Optional[int], file: str, row_idx: int) -> Optional[Dict[str, Any]]:
    """
    支持两种 Unsold 文件：
    A) BrandYYYYUnsold.csv  -> 收货年份来自 filename
    B) BrandUnsold.csv      -> 收货年份来自每行第 4 列（index=3）
    """
    while len(cols) < 10:
        cols.append("")

    recv_token = cols[0]
    name = cols[1]
    cost_rmb = to_int_maybe(cols[2])

    # ⭐ 行内年份（你给的例子：第 4 列就是 2023/2024）
    row_year = to_int_maybe(cols[3])

    # ⭐ 最终使用的收货年份：优先 filename，其次行内
    recv_year_use = recv_year_from_filename if recv_year_from_filename is not None else row_year

    if recv_year_use is None:
        print(f"[WARN] {file} row#{row_idx}: missing recv year (filename has no year AND row year empty) -> skip")
        return None

    dm = parse_recv_day_month(recv_token)
    if not dm:
        print(f"[WARN] {file} row#{row_idx}: invalid recv date token: {recv_token!r} -> skip")
        return None

    r_day, r_month = dm

    received_at = safe_dt_utc(int(recv_year_use), r_month, r_day)
    if not received_at:
        print(f"[WARN] {file} row#{row_idx}: invalid received date -> year={recv_year_use} token={recv_token!r} parsed={r_day}.{r_month} -> skip")
        return None

    serial = (cols[7] or "").strip() or None
    note = concat_notes(cols[8], cols[9])

    item_doc = {
        "name": name.strip(),
        "brand": brand,
        "currency": "RMB",
        "cost": int(cost_rmb or 0),

        "sell_price": 0,
        "sell_currency": "",
        "profit": 0,
        "profit_currency": "",
        "fee": 0,

        "status": "RECEIVED",
        "note": note,
        "serial_code": serial,

        "code": "",
        "accessories": "",
        "seller_name": "",
        "seller_contact": "",

        "source_type": "BUY_IN",
        "is_buy_in": True,
        "is_consignment": False,

        "created_at": received_at,
        "updated_at": received_at,
        "received_at": received_at,
        "sold_at": None,

        "audit": [{
            "at": datetime.now(timezone.utc),
            "action": "IMPORT_LEGACY",
            "detail": {"file": file, "row": row_idx, "kind": "Unsold"}
        }]
    }

    print(f"[OK] {file} row#{row_idx} | {brand} | recv={received_at.date()} | UNSOLD | costRMB={cost_rmb} | serial={serial or '-'} | name={name[:30]}...")
    return {"item": item_doc, "sale": None, "meta": {"file": file, "row": row_idx, "kind": "Unsold"}}


# %% [markdown]
# ## 扫描目录、解析所有文件、打印每条、导出 payload

# %%
all_records: List[Dict[str, Any]] = []

if not DATA_DIR.exists():
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR.resolve()}")

files = sorted([p for p in DATA_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".csv"])
print("Found files:", len(files))

for fp in files:
    m1 = FILENAME_RE_YEAR.match(fp.name)
    m2 = FILENAME_RE_NOYEAR_UNSOLD.match(fp.name)

    recv_year_from_filename = None
    kind = None
    brand_raw = None

    if m1:
        brand_raw = m1.group("brand")
        recv_year_from_filename = int(m1.group("year"))
        kind = m1.group("kind").capitalize()  # Sold / Unsold
    elif m2:
        brand_raw = m2.group("brand")
        kind = "Unsold"
        recv_year_from_filename = None  # 这一类要从行内取年份
    else:
        print(f"[SKIP] filename not matched pattern: {fp.name}")
        continue

    brand = normalize_brand(brand_raw)

    rows = sniff_rows(fp)
    if not rows:
        print(f"[WARN] empty file: {fp.name}")
        continue

    # 有些 csv 可能有表头，我们简单判定：如果第一行第一列不是日期 token，就跳过第一行
    start_idx = 0
    if rows and (not parse_recv_day_month(rows[0][0] if rows[0] else "")):
        start_idx = 1

    for i, cols in enumerate(rows[start_idx:], start=start_idx):
        # 跳过空行
        if not cols or all(str(c).strip() == "" for c in cols):
            continue

        if kind == "Sold":
            rec = parse_sold_row(cols, brand, recv_year_from_filename, fp.name, i)
        else:
            rec = parse_unsold_row(cols, brand, recv_year_from_filename, fp.name, i)


        if rec:
            all_records.append(rec)

print("\nTotal parsed records:", len(all_records))

# %%
# 导出 payload（一次性写出给 Notebook2 用）
with OUT_PAYLOAD.open("wb") as f:
    pickle.dump(all_records, f)

print("Saved payload to:", OUT_PAYLOAD.resolve())


Found files: 19
[OK] Chanel2022Sold.csv row#0 | Chanel | recv=2022-05-26 | sold=2022-06-18 | costRMB=9900 sellSGD=2450 profitRMB=1760 | serial=- | name=GST SHW fanny...
[OK] Chanel2022Sold.csv row#1 | Chanel | recv=2022-05-26 | sold=2022-06-20 | costRMB=25000 sellSGD=5550 profitRMB=1540 | serial=- | name=CF Jumbo SHW fanny...
[OK] Chanel2022Sold.csv row#2 | Chanel | recv=2022-06-12 | sold=2022-07-03 | costRMB=18800 sellSGD=5750 profitRMB=8500 | serial=- | name=CF Medium Red银（护理）fanny...
[OK] Chanel2022Sold.csv row#3 | Chanel | recv=2022-06-13 | sold=2022-07-03 | costRMB=9999 sellSGD=1800 profitRMB=-1400 | serial=- | name=GST GHW fanny...
[OK] Chanel2022Sold.csv row#4 | Chanel | recv=2022-06-15 | sold=2022-07-04 | costRMB=10500 sellSGD=2200 profitRMB=0 | serial=- | name=GST SHW fanny...
[OK] Chanel2022Sold.csv row#5 | Chanel | recv=2022-06-15 | sold=2022-07-09 | costRMB=9999 sellSGD=2650 profitRMB=2525 | serial=- | name=GST GHW fanny...
[OK] Chanel2022Sold.csv row#6 | Chanel | recv=2022

In [13]:
# %% [markdown]
# # Legacy CSV Import - Step 2: Write to MongoDB in one batch
#
# 读取：
#   ./import_payload.pkl
# 写入：
#   MongoDB: items + sales
#
# 注意：
# - SKU 在这里生成，保证不重复（会检查 DB）
# - 所有数据准备完成后才 insert_many（一次性写入）

# %%
import os
import re
import secrets
import pickle
from pathlib import Path
from datetime import timezone, datetime
from typing import List, Dict, Any

from pymongo import MongoClient, ASCENDING
from bson import ObjectId

PAYLOAD_PATH = Path("./import_payload.pkl")

MONGO_URI = os.getenv("MONGODB_URI", "mongodb://localhost:27017/")
DB_NAME = os.getenv("INV_DB", "inventory")           # 你按实际库名改（例如 inventory_app）
ITEMS_COLL = os.getenv("INV_ITEMS", "items")
SALES_COLL = os.getenv("INV_SALES", "sales")

# ===== SKU 生成：品牌前缀 + 7位短码（Base32，短且不含易混淆字符）=====
_BASE32_ALPHABET = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"

def _short_code(n=7) -> str:
    return "".join(secrets.choice(_BASE32_ALPHABET) for _ in range(n))

def _brand_prefix(brand: str) -> str:
    b = (brand or "").strip()
    if not b:
        return "ITEM"

    mapping = {
        "Louis Vuitton": "LV",
        "Van Cleef & Arpels": "VCA",
        "Saint Laurent": "YSL",
        "Bottega Veneta": "BV",
    }
    if b in mapping:
        return mapping[b]

    cleaned = re.sub(r"[^A-Za-z0-9]+", "", b).upper()
    return cleaned[:8] if cleaned else "ITEM"

def gen_sku_unique(items_coll, brand: str) -> str:
    prefix = _brand_prefix(brand)
    for _ in range(30):
        sku = f"{_short_code(7)}"
        if items_coll.count_documents({"sku": sku}, limit=1) == 0:
            return sku
    return f"{_short_code(9)}"

# %%
if not PAYLOAD_PATH.exists():
    raise FileNotFoundError(f"payload not found: {PAYLOAD_PATH.resolve()}")

with PAYLOAD_PATH.open("rb") as f:
    payload: List[Dict[str, Any]] = pickle.load(f)

print("Loaded payload records:", len(payload))

# %%
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
items = db[ITEMS_COLL]
sales = db[SALES_COLL]

# 一些必要索引（没有也能写，但建议）
items.create_index([("sku", ASCENDING)], unique=True, sparse=True)
sales.create_index([("item_id", ASCENDING)], unique=True, sparse=True)

print("Connected to DB:", DB_NAME, "| collections:", ITEMS_COLL, SALES_COLL)

# %%
# 先在内存里构造最终要写入的 docs
item_docs: List[Dict[str, Any]] = []
sale_docs: List[Dict[str, Any]] = []

for idx, rec in enumerate(payload, start=1):
    item_doc = rec["item"]
    sale_doc = rec["sale"]

    # 生成 SKU
    sku = gen_sku_unique(items, item_doc.get("brand", ""))
    item_doc["sku"] = sku

    # 为了让你的图片系统后续可用（./photos/[sku]/...），sku 一旦写入就固定
    # 可选：将 legacy serial 放到 code 里（如果你更希望 code=serial，可以自己改）
    # item_doc["code"] = item_doc.get("serial_code") or ""

    # 插入 item 之前先生成 _id（用于 sale 关联）
    item_id = ObjectId()
    item_doc["_id"] = item_id

    item_docs.append(item_doc)

    if sale_doc is not None:
        sale_doc["item_id"] = str(item_id)  # 你当前 sales 表设计是字符串 item_id（按你之前代码）
        sale_doc["sku"] = sku
        sale_docs.append(sale_doc)

    if idx % 200 == 0:
        print(f"Prepared {idx}/{len(payload)} ...")

print("Prepared items:", len(item_docs), "| sales:", len(sale_docs))

# %%
# ===== 一次性写入（先 items 再 sales）=====
# 如果你担心误写，先把下面 insert_many 注释掉跑一次看看 Prepared 打印即可。

if item_docs:
    items.insert_many(item_docs, ordered=True)
    print("Inserted items:", len(item_docs))

if sale_docs:
    # sales.item_id 有 unique index 时，重复会报错；这里默认不会重复
    sales.insert_many(sale_docs, ordered=True)
    print("Inserted sales:", len(sale_docs))

print("DONE.")


Loaded payload records: 5489
Connected to DB: inventory | collections: items sales
Prepared 200/5489 ...
Prepared 400/5489 ...
Prepared 600/5489 ...
Prepared 800/5489 ...
Prepared 1000/5489 ...
Prepared 1200/5489 ...
Prepared 1400/5489 ...
Prepared 1600/5489 ...
Prepared 1800/5489 ...
Prepared 2000/5489 ...
Prepared 2200/5489 ...
Prepared 2400/5489 ...
Prepared 2600/5489 ...
Prepared 2800/5489 ...
Prepared 3000/5489 ...
Prepared 3200/5489 ...
Prepared 3400/5489 ...
Prepared 3600/5489 ...
Prepared 3800/5489 ...
Prepared 4000/5489 ...
Prepared 4200/5489 ...
Prepared 4400/5489 ...
Prepared 4600/5489 ...
Prepared 4800/5489 ...
Prepared 5000/5489 ...
Prepared 5200/5489 ...
Prepared 5400/5489 ...
Prepared items: 5489 | sales: 4621
Inserted items: 5489
Inserted sales: 4621
DONE.
